# FraudSAGE — Evaluation & Embedding Space Visualization

Loads saved artifacts from `resources/outputs/` and produces:
1. **UMAP embedding space** with GMM cluster background colours and true-label overlays
2. **PR-AUC curve** and **Top-K Lift** for the final pipeline score (`scarcity_anchor_ensemble_prob`)

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, to_rgba
from pathlib import Path

OUTPUTS = Path("resources/outputs")

# ── Load artifacts ─────────────────────────────────────────────────────────
embeddings  = np.load(OUTPUTS / "dgi_customer_embeddings.npy")
gmm_model   = joblib.load(OUTPUTS / "dgi_gmm.joblib")
comp_df     = pd.read_csv(OUTPUTS / "dgi_component_assignments.csv.gz")
comp_stats  = pd.read_csv(OUTPUTS / "dgi_gmm_component_stats.csv")

# Pick best available score source
for _fname, _col in [
    ("model_output.csv",                   "fraud_score"),
    ("dgi_mlp_customer_scores.csv.gz",     "mlp_fraud_prob"),
    ("dgi_customer_scores.csv.gz",         "dgi_anomaly_score"),
]:
    _p = OUTPUTS / _fname
    if _p.exists():
        scores_df = pd.read_csv(_p)
        score_col = _col if _col in scores_df.columns else scores_df.columns[-1]
        print(f"Score source: {_fname}  →  column '{score_col}'")
        break

print(f"Embeddings : {embeddings.shape}")
print(f"Customers  : {len(comp_df):,}  |  GMM components: {gmm_model.n_components}")

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
import umap
from sklearn.metrics import precision_recall_curve, auc, average_precision_score
from pathlib import Path

OUTPUTS = Path("resources/outputs")

# ── Load artifacts ─────────────────────────────────────────────────────────
embeddings    = np.load(OUTPUTS / "dgi_customer_embeddings.npy")
gmm_model     = joblib.load(OUTPUTS / "dgi_gmm.joblib")
comp_df       = pd.read_csv(OUTPUTS / "dgi_component_assignments.csv.gz")
comp_stats    = pd.read_csv(OUTPUTS / "dgi_gmm_component_stats.csv")

# model_output may not exist yet if cell 33 hasn't run — try both score sources
model_out_path = OUTPUTS / "model_output.csv"
mlp_path       = OUTPUTS / "dgi_mlp_customer_scores.csv.gz" 

if model_out_path.exists():
    scores_df = pd.read_csv(model_out_path)
    score_col = "fraud_score"         # scarcity_anchor_ensemble_prob renamed
elif mlp_path.exists():
    scores_df = pd.read_csv(mlp_path)
    score_col = "mlp_fraud_prob"
else:
    raise FileNotFoundError("Run training cells 25–33 first to generate score files.")

print(f"Embeddings : {embeddings.shape}")
print(f"Component df: {len(comp_df):,} rows")
print(f"Score source: {score_col}  ({len(scores_df):,} rows)")

In [ ]:

# ── Load & merge all artifacts ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

OUTPUTS = Path("resources/outputs")

embeddings  = np.load(OUTPUTS / "dgi_customer_embeddings.npy")
gmm_model   = joblib.load(OUTPUTS / "dgi_gmm.joblib")
comp_df     = pd.read_csv(OUTPUTS / "dgi_component_assignments.csv.gz")
comp_stats  = pd.read_csv(OUTPUTS / "dgi_gmm_component_stats.csv")
scores_df   = pd.read_csv(OUTPUTS / "model_output.csv")

SCORE_COL = "scarcity_anchor_ensemble_prob"

merged = comp_df.merge(
    scores_df[["customer_id", SCORE_COL, "fraud_score"]],
    on="customer_id", how="left",
)
fraud_comp_set = set(comp_stats.loc[comp_stats["fraud_component"] == True, "component"])
merged["is_fraud_component"] = merged["component"].isin(fraud_comp_set)

gold = merged[merged["is_labeled"] == 1].copy()
print(f"Total customers : {len(merged):,}")
print(f"Gold labeled    : {len(gold):,}  "
      f"(fraud={int(gold['true_label'].sum())}, "
      f"legit={int((gold['true_label']==0).sum())})")
print(f"Fraud components: {len(fraud_comp_set)} / {gmm_model.n_components}")
print(f"Score column    : {SCORE_COL}")


In [ ]:

# ── PR-AUC and Top-K Lift (gold-labeled customers) ───────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc, average_precision_score

y_true  = gold["true_label"].to_numpy(dtype=int)
y_score = gold[SCORE_COL].to_numpy(dtype=float)

precision_arr, recall_arr, _ = precision_recall_curve(y_true, y_score)
pr_auc   = auc(recall_arr, precision_arr)
ap       = average_precision_score(y_true, y_score)
baseline = y_true.mean()

order     = np.argsort(y_score)[::-1]
y_sorted  = y_true[order]
cum_fraud = np.cumsum(y_sorted)
ks        = np.arange(1, len(y_sorted) + 1)
lift      = (cum_fraud / ks) / baseline
pcts      = ks / len(y_sorted) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("FraudSAGE — Pipeline Evaluation (Gold-Labeled Customers)", fontsize=14, fontweight="bold")

# ── PR curve ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(recall_arr, precision_arr, color="#e63946", lw=2, label=f"PR-AUC = {pr_auc:.3f}")
ax.axhline(baseline, color="grey", ls="--", lw=1.2, label=f"Random ({baseline:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curve"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# ── Top-K Lift ────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(pcts, lift, color="#457b9d", lw=2)
ax.axhline(1.0, color="grey", ls="--", lw=1.2)

# Only annotate well-spaced points to avoid overlap
annot_k   = [5, 20, 50]
annot_off = [(10, 0.4), (30, 0.4), (62, 0.4)]
colors    = ["#e76f51", "#2a9d8f", "#9b59b6"]
for k_pct, (dx, dy), col in zip(annot_k, annot_off, colors):
    idx = max(0, int(k_pct / 100 * len(y_sorted)) - 1)
    lv  = lift[idx]
    ax.scatter(pcts[idx], lv, color=col, zorder=5, s=70)
    ax.annotate(
        f"Top {k_pct}%  ×{lv:.1f}",
        xy=(pcts[idx], lv),
        xytext=(pcts[idx] + dx, lv + dy),
        fontsize=9, color=col,
        arrowprops=dict(arrowstyle="-", lw=0.9, color=col, alpha=0.8),
    )

# Summary table (top-right inset)
table_ks  = [1, 5, 10, 20, 50]
table_txt = "\n".join(
    [f"  Top {k:>2}%  ×{lift[max(0, int(k/100*len(y_sorted))-1)]:.2f}" for k in table_ks]
)
ax.text(0.97, 0.97, "Lift summary\n" + table_txt,
        transform=ax.transAxes, fontsize=8.5, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="#ccc"),
        family="monospace")

ax.set_xlabel("Top-K % of customers ranked by score"); ax.set_ylabel("Lift over random")
ax.set_title("Top-K Precision Lift"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pr_auc_topk_lift.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PR-AUC: {pr_auc:.4f}   AP: {ap:.4f}")
for k in [1, 5, 10, 20]:
    idx = max(0, int(k / 100 * len(y_sorted)) - 1)
    print(f"  Top {k:>2}% lift: ×{lift[idx]:.2f}")


In [ ]:

# ── t-SNE + GMM Cluster Visualization ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import colorsys
import openTSNE
from sklearn.decomposition import PCA

# ── 1. Fit t-SNE (PCA init for stability) ────────────────────────────────────
print("Fitting t-SNE (this may take ~3 min)...")
pca_init = PCA(n_components=2, random_state=42).fit_transform(embeddings)
pca_init /= np.std(pca_init[:, 0]) * 10000

tsne = openTSNE.TSNE(
    perplexity=50,
    n_iter=750,
    metric="cosine",
    initialization=pca_init,
    n_jobs=-1,
    random_state=42,
    verbose=False,
)
tsne_xy = tsne.fit(embeddings)
print("t-SNE done.")

# ── 2. Per-component colour palette ──────────────────────────────────────────
n_comp = gmm_model.n_components
rng    = np.random.default_rng(7)
hues   = np.linspace(0, 1, n_comp, endpoint=False)
rng.shuffle(hues)

def hsva(h, s=0.65, v=0.90, a=1.0):
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (r, g, b, a)

comp_rgba   = np.array([hsva(h, a=0.55) for h in hues])
comp_labels = merged["component"].to_numpy()
pt_colors   = comp_rgba[comp_labels]

# ── 3. Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 10))
ax.set_facecolor("#0e1117"); fig.patch.set_facecolor("#0e1117")

non_fraud_mask  = ~np.isin(comp_labels, list(fraud_comp_set))
fraud_comp_mask =  np.isin(comp_labels, list(fraud_comp_set))

ax.scatter(tsne_xy[non_fraud_mask, 0], tsne_xy[non_fraud_mask, 1],
           c=pt_colors[non_fraud_mask], s=3, linewidths=0, zorder=2, alpha=0.65)
ax.scatter(tsne_xy[fraud_comp_mask, 0], tsne_xy[fraud_comp_mask, 1],
           c="#ff6b6b", s=5, linewidths=0, zorder=3, alpha=0.80)

# Gold labels as hollow rings — cluster colour stays visible underneath
y_all   = merged["true_label"].to_numpy(dtype=float)
labeled = merged["is_labeled"].to_numpy() == 1
legit_m = labeled & (y_all == 0)
fraud_m = labeled & (y_all == 1)

ax.scatter(tsne_xy[legit_m, 0], tsne_xy[legit_m, 1],
           facecolors="none", edgecolors="#4cc9f0", linewidths=1.1, s=55, zorder=6)
ax.scatter(tsne_xy[fraud_m, 0], tsne_xy[fraud_m, 1],
           facecolors="none", edgecolors="#f72585", linewidths=1.6, s=70, zorder=7)

legend_elems = [
    mpatches.Patch(facecolor="#888888", alpha=0.7,
                   label=f"Non-fraud GMM components ({n_comp - len(fraud_comp_set)})"),
    mpatches.Patch(facecolor="#ff6b6b", alpha=0.8,
                   label=f"Fraud-associated components ({len(fraud_comp_set)})"),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
               markeredgecolor='#4cc9f0', markersize=9,
               label=f"Gold Legit ({legit_m.sum():,})"),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
               markeredgecolor='#f72585', markersize=9, markeredgewidth=1.6,
               label=f"Gold Fraud ({fraud_m.sum():,})"),
]
ax.legend(handles=legend_elems, loc="upper right", framealpha=0.45,
          fontsize=11, facecolor="#1a1a2e", labelcolor="white")

ax.set_title(
    "FraudSAGE — DGI Customer Embedding Space (t-SNE)\n"
    "Each colour = one GMM component  |  Red = fraud-associated  |  Hollow rings = gold labels",
    color="white", fontsize=13, pad=12)
ax.tick_params(colors="white")
for sp in ax.spines.values():
    sp.set_edgecolor("#444")
ax.set_xlabel("t-SNE 1", color="white", fontsize=11)
ax.set_ylabel("t-SNE 2", color="white", fontsize=11)

plt.tight_layout()
plt.savefig("tsne_gmm_clusters.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved → tsne_gmm_clusters.png")
